In [1]:
import json
from kafka import KafkaConsumer, KafkaProducer
import random

#Topic for getting the data fro inference
INPUT_TOPIC = "dummy-windowed"
#New to integrate
# Topic for the grafana. you can add more as is shown below
OUTPUT_TOPIC = "dummy-aggregated"
BOOTSTRAP_SERVERS = ["kafka.apache-kafka.svc.cluster.local:9092"]

# Consumer your already have it
consumer = KafkaConsumer(
    INPUT_TOPIC,
    bootstrap_servers=BOOTSTRAP_SERVERS,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    auto_offset_reset="latest",
    enable_auto_commit=True,
    group_id="sum-computation-group"
)

# New to integrate: Add Producer
producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

print(f"Listening on topic {INPUT_TOPIC} and publishing to {OUTPUT_TOPIC}")

for msg in consumer:
    data = msg.value

    values = []
    ts = None

    if "inputs" in data:
        for item in data["inputs"]:
            if isinstance(item, list) and len(item) > 0:
                entry = item[0]
                val = entry.get("value")
                if val is not None:
                    values.append(val)
                if ts is None:  # take timestamp from first element
                    ts = entry.get("timestamp")
    #total = forecasted value
    forecasted_value = sum(values)
    real_value= random.randint(0, 5)

    #only to print the output
    result = {
        "timestamp": ts,
        "inference": round(forecasted_value, 6),
        "real-value": real_value
    }
     
    ##New to integrate
    ##Publish the data in a new topic . you can group several metrics yo see in ine grafana to avoid creating more topics
    producer.send(OUTPUT_TOPIC, value=result)
    print("Published:", result)
    
    ##Publish the data in a new topic if oyu wan to add more 
    #producer.send("real_value", value=result)
    #print("Published:", result)


Listening on topic dummy-windowed and publishing to dummy-aggregated


KeyboardInterrupt: 